# MiVOLO v2 and YOLOv8 conversion with OpenVINO and NNCF

Convert MiVOLO v2 estimator and YOLOv8 person/face detector to static
FP16-weight OpenVINO IR, quantize MiVOLO with NNCF accuracy control, and validate
conversion and quantization. FP16 weight compression does not force FP16 execution
on every device; INT8 is mixed precision, not an all-INT8 graph.

The original PyTorch forward implementations are exported without custom graph
rewrites. See [README](README.md) for installation and validation limitations.

## Table of contents

- [Installation instructions](#Installation-instructions)
- [FP16 conversion](#1.-Convert-MiVOLO-and-YOLOv8-to-FP16-weight-IR)
- [INT8 conversion](#2.-Accuracy-controlled-MiVOLO-INT8-conversion)
- [Conversion validation](#3.-Validate-conversion-against-original-PyTorch-models)
- [INT8 accuracy validation](#4.-Validate-INT8-on-held-out-FairFace-data)
- [Interactive visualization](#5.-Interactive-age-and-gender-visualization)
- [Latency and throughput benchmarks](#6.-FP16-versus-INT8-latency-and-throughput)

## Installation instructions

This is a self-contained example. We recommend running it in a dedicated virtual
environment with Jupyter available.

For general environment setup, see the main
[OpenVINO Notebooks Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md).

Open [mivolo-v2.ipynb](mivolo-v2.ipynb) and run all cells in order.

In [1]:
# Pinned to the versions this notebook was validated with. MiVOLO's setup.py still imports pkg_resources,
# which setuptools 81 removed, so it is built without build isolation against an older setuptools.
# omegaconf is needed to unpickle the YOLO checkpoint; pyarrow/pandas are pinned to builds that work with numpy 1.x.
%pip install -q "setuptools<81"
%pip install -q "matplotlib==3.8.4" "ipywidgets==8.1.9"
%pip install -q --no-build-isolation "mivolo @ git+https://github.com/WildChlamydia/MiVOLO.git@37475e3f8818b5f22448003feec3e64b01bfb188" "openvino==2026.4.0" "nncf==3.4.0" "torch==2.14.0" "torchvision==0.29.0" "transformers==5.17.0" "datasets==5.0.1" "pyarrow==25.0.1" "pandas==3.0.5" "huggingface_hub==1.31.0" "ultralytics==8.1.0" "omegaconf==2.3.1" "onnx==1.23.0" "onnxscript==0.7.2" "opencv-python==4.9.0.80" "numpy==1.26.4" --extra-index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import tempfile
from pathlib import Path
import cv2
import numpy as np
import openvino as ov
import nncf
import torch
from transformers import AutoConfig, AutoModelForImageClassification
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Setup directory paths
BASE_DIR = Path("mivolo_workspace")
MODELS_FP16_DIR = BASE_DIR / "models_fp16"
MODELS_INT8_DIR = BASE_DIR / "models_int8"
for p in [MODELS_FP16_DIR, MODELS_INT8_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Hugging Face repos are pinned to exact commits: MiVOLO loads remote code (trust_remote_code) and the YOLO
# checkpoint is a pickle, so both must come from reviewed, immutable revisions.
MIVOLO_MODEL_ID = "iitolstykh/mivolo_v2"
MIVOLO_REVISION = "53393526c220e34cdd7b722b36d22b6f9e5f4241"
YOLO_REPO_ID = "iitolstykh/YOLO-Face-Person-Detector"
YOLO_REVISION = "b3d071aedccd46a3b2d4b40609da6880a815f395"
FAIRFACE_DATASET_ID = "HuggingFaceM4/FairFace"
FAIRFACE_REVISION = "54d573cdb8b5af490ba8da9da2799628f6e5c496"

MIVOLO_CONFIG = AutoConfig.from_pretrained(MIVOLO_MODEL_ID, revision=MIVOLO_REVISION, trust_remote_code=True)
MIVOLO_NUM_GENDER_CLASSES = int(MIVOLO_CONFIG.num_classes_gender)
MIVOLO_MIN_AGE = float(MIVOLO_CONFIG.min_age)
MIVOLO_MAX_AGE = float(MIVOLO_CONFIG.max_age)
MIVOLO_AVG_AGE = float(MIVOLO_CONFIG.avg_age)
MIVOLO_PRIMARY_INPUT_SIZE = 384

MIVOLO_FP16_PATH = MODELS_FP16_DIR / "mivolo_v2.xml"
MIVOLO_INT8_PATH = MODELS_INT8_DIR / "mivolo_v2.xml"
YOLO_FP16_PATH = MODELS_FP16_DIR / "yolov8x_person_face.xml"

core = ov.Core()
# Set MIVOLO_DEVICE to CPU, GPU, NPU or AUTO before running.
DEFAULT_DEVICE = os.environ.get("MIVOLO_DEVICE", "GPU")
print(f"OpenVINO version: {ov.__version__}")
print(f"Available execution devices: {core.available_devices}")
print(f"Default device: {DEFAULT_DEVICE}")

def compile_for_device(core, model, device):
    compiled = core.compile_model(model, device)
    print(f"{device} execution devices: {compiled.get_property('EXECUTION_DEVICES')}")
    return compiled

## 1. Convert MiVOLO and YOLOv8 to FP16-weight IR

Both exports use fixed batch-one shapes and the original model forward implementations.
Only standard ONNX export and OpenVINO conversion are applied; there are no custom
forward patches or post-conversion IR transformations.

In [ ]:
MIVOLO_EXPORT_REVISION_KEY = "mivolo_conversion_revision"
MIVOLO_EXPORT_REVISION = "original-forward-onnx18-v1"


def needs_mivolo_export(model_path):
    if not model_path.exists():
        return True

    try:
        existing_mivolo = core.read_model(model_path)
        existing_inputs = {tensor.get_any_name() for tensor in existing_mivolo.inputs}
        if existing_inputs != {"concat_input"}:
            return True
        # Do not reuse IRs generated by a different conversion workflow.
        if not existing_mivolo.has_rt_info(MIVOLO_EXPORT_REVISION_KEY):
            return True
        if existing_mivolo.get_rt_info(MIVOLO_EXPORT_REVISION_KEY).astype(str) != MIVOLO_EXPORT_REVISION:
            return True
    except RuntimeError as error:
        print(f"Cannot read cached FP16 IR; rebuilding: {error}")
        return True

    return False

# --- 1. Convert MiVOLO v2 ---
class MiVOLOHeadExportWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, concat_input):
        return self.model.mivolo(concat_input=concat_input, return_dict=True).head_outputs


def load_mivolo_for_export():
    model = AutoModelForImageClassification.from_pretrained(
        MIVOLO_MODEL_ID,
        revision=MIVOLO_REVISION,
        trust_remote_code=True,
        dtype=torch.float32,
    )
    return model.eval()


def ensure_mivolo_fp16_model():
    # Set FORCE_MIVOLO_EXPORT=1 to rebuild even when the cached IR is current.
    force_rebuild = os.environ.get("FORCE_MIVOLO_EXPORT", "0") == "1"
    if not force_rebuild and not needs_mivolo_export(MIVOLO_FP16_PATH):
        print(f"MiVOLO FP16 IR is up to date: {MIVOLO_FP16_PATH}")
        return
    print("Loading MiVOLO v2 from Hugging Face Hub...")
    export_model = MiVOLOHeadExportWrapper(load_mivolo_for_export())
    dummy_concat = torch.randn(1, 6, MIVOLO_PRIMARY_INPUT_SIZE, MIVOLO_PRIMARY_INPUT_SIZE, dtype=torch.float32)

    print(f"Exporting MiVOLO v2 to static ONNX and OpenVINO IR for {MIVOLO_PRIMARY_INPUT_SIZE}x{MIVOLO_PRIMARY_INPUT_SIZE}...")
    with tempfile.TemporaryDirectory() as staging_dir:
        onnx_path = Path(staging_dir) / "mivolo_v2.onnx"
        torch.onnx.export(
            export_model,
            (dummy_concat,),
            onnx_path.as_posix(),
            input_names=["concat_input"],
            output_names=["head_outputs"],
            opset_version=18,
            do_constant_folding=True,
            dynamic_axes=None,
            training=torch.onnx.TrainingMode.EVAL,
        )
        ov_mivolo = ov.convert_model(onnx_path.as_posix())
        # Save before the staging dir is removed: the converted model may still map the ONNX weight file.
        ov_mivolo.set_rt_info(MIVOLO_EXPORT_REVISION, MIVOLO_EXPORT_REVISION_KEY)
        ov.save_model(ov_mivolo, MIVOLO_FP16_PATH, compress_to_fp16=True)
        del ov_mivolo
    del export_model
    print(f"Saved: {MIVOLO_FP16_PATH}")


ensure_mivolo_fp16_model()

# --- 2. Convert YOLOv8x Detector ---
if not YOLO_FP16_PATH.exists():
    print("Downloading YOLOv8x Person/Face model from Hugging Face Hub...")
    pt_local_path = hf_hub_download(repo_id=YOLO_REPO_ID, filename="yolov8x_person_face.pt", revision=YOLO_REVISION)

    # Ultralytics checkpoints are full pickles, which torch.load refuses by default; this is only acceptable
    # because the file comes from the pinned YOLO_REVISION above.
    print("Loading PyTorch model...")
    _orig_torch_load = torch.load
    try:
        torch.load = lambda *args, **kwargs: _orig_torch_load(*args, **{**kwargs, "weights_only": False})
        yolo_model = YOLO(pt_local_path)
    finally:
        torch.load = _orig_torch_load

    # Export the pinned detector to a static ONNX intermediate.
    print("Exporting YOLOv8x to ONNX...")
    # opset 18 is what the torch exporter emits; requesting older opsets only triggers a failed down-conversion.
    onnx_path_str = yolo_model.export(format="onnx", imgsz=640, opset=18)

    # Step 2.2: Convert ONNX to OpenVINO IR 
    print("Converting YOLOv8x ONNX to OpenVINO IR...")
    ov_yolo = ov.convert_model(onnx_path_str)

    # Save the final FP16 model
    ov.save_model(ov_yolo, YOLO_FP16_PATH, compress_to_fp16=True)
    del ov_yolo, yolo_model
    print(f"Saved: {YOLO_FP16_PATH}")
else:
    print(f"YOLOv8x IR already exists at {YOLO_FP16_PATH}")

## 2. Accuracy-controlled MiVOLO INT8 conversion

Calibrate on 300 FairFace training samples and use the next 150 disjoint training
samples for NNCF accuracy control (maximum age MAE increase: 0.75 years).
NNCF determines which operations to retain in floating point through accuracy control;
no graph-specific attention exclusions are applied. YOLO remains FP16; no detector
INT8 calibration is claimed.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
FAIRFACE_AGE_BUCKET_MIDPOINTS = {0: 2, 1: 6, 2: 15, 3: 25, 4: 35, 5: 45, 6: 55, 7: 65, 8: 75}


def load_fairface_samples(split, start, count):
    # Yields (face_bgr, body_bgr, gender_idx, age_years) for `count` usable samples, skipping the first
    # `start` usable ones, so calibration and validation can use disjoint slices of the same split.
    # FairFace images are face-centered, so the face crop is a fixed center region and the body is the image.
    dataset = load_dataset(FAIRFACE_DATASET_ID, "0.25", split=split, streaming=True, revision=FAIRFACE_REVISION)
    usable = 0
    for item in dataset:
        bgr_img = cv2.cvtColor(np.array(item["image"]), cv2.COLOR_RGB2BGR)
        h, w = bgr_img.shape[:2]
        face_crop = bgr_img[int(h * 0.1):int(h * 0.5), int(w * 0.25):int(w * 0.75)]
        if face_crop.size == 0:
            continue
        usable += 1
        if usable <= start:
            continue
        gender = item["gender"]
        if gender not in (0, 1, "Male", "Female"):
            raise ValueError(f"Unsupported FairFace gender: {gender!r}")
        gender_idx = 0 if gender in (0, "Male") else 1
        age_years = float(FAIRFACE_AGE_BUCKET_MIDPOINTS[item["age"]])
        yield face_crop, bgr_img, gender_idx, age_years
        if usable >= start + count:
            return
    raise ValueError(f"Requested {count} samples after {start}, found only {max(0, usable - start)}")


def preprocess_crop(crop_img):
    """Prepare float NCHW RGB data for calibration and validation."""
    resized = cv2.resize(crop_img, (MIVOLO_PRIMARY_INPUT_SIZE, MIVOLO_PRIMARY_INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    norm = (rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    tensor = np.transpose(norm, (2, 0, 1))
    return np.ascontiguousarray(np.expand_dims(tensor, axis=0), dtype=np.float32)


def head_outputs_to_age(head_outputs):
    return head_outputs[:, MIVOLO_NUM_GENDER_CLASSES] * (MIVOLO_MAX_AGE - MIVOLO_MIN_AGE) + MIVOLO_AVG_AGE

MIVOLO_INT8_REVISION_KEY = "mivolo_int8_revision"
MIVOLO_INT8_REVISION = "original-forward-accuracy-control-v1"

def needs_mivolo_int8_rebuild():
    # An INT8 IR quantized from an older FP16 export inherits that export's graph, so it is rebuilt with it.
    if not MIVOLO_INT8_PATH.exists():
        return True
    try:
        existing_int8 = core.read_model(MIVOLO_INT8_PATH)
        if not existing_int8.has_rt_info(MIVOLO_INT8_REVISION_KEY):
            return True
        if existing_int8.get_rt_info(MIVOLO_INT8_REVISION_KEY).astype(str) != MIVOLO_INT8_REVISION:
            return True
    except RuntimeError as error:
        print(f"Cannot read cached INT8 IR; rebuilding: {error}")
        return True
    return MIVOLO_INT8_PATH.stat().st_mtime < MIVOLO_FP16_PATH.stat().st_mtime


if needs_mivolo_int8_rebuild():
    print(f"Streaming FairFace samples from Hugging Face ('{FAIRFACE_DATASET_ID}')...")
    calibration_samples = [
        np.concatenate([preprocess_crop(face), preprocess_crop(body)], axis=1)
        for face, body, _, _ in load_fairface_samples("train", start=0, count=300)
    ]
    # Accuracy control is judged on held-out images that calibration never saw.
    validation_samples = [
        (np.concatenate([preprocess_crop(face), preprocess_crop(body)], axis=1), age)
        for face, body, _, age in load_fairface_samples("train", start=300, count=150)
    ]

    def validation_fn(compiled_model, validation_loader):
        age_errors = [
            abs(float(head_outputs_to_age(compiled_model(concat_input)[compiled_model.output("head_outputs")])[0]) - age)
            for concat_input, age in validation_loader
        ]
        # NNCF treats the metric as higher-is-better, so report negative MAE: max_drop then caps the MAE increase.
        return -np.mean(age_errors)

    ov_mivolo_fp16 = core.read_model(MIVOLO_FP16_PATH)
    print("Quantizing MiVOLO v2 with NNCF using accuracy-aware optimization...")
    quantized_model = nncf.quantize_with_accuracy_control(
        model=ov_mivolo_fp16,
        calibration_dataset=nncf.Dataset(calibration_samples, lambda concat_input: {"concat_input": concat_input}),
        validation_dataset=nncf.Dataset(validation_samples, lambda sample: {"concat_input": sample[0]}),
        validation_fn=validation_fn,
        max_drop=0.75,  # Max allowable increase in age MAE (years)
        preset=nncf.QuantizationPreset.PERFORMANCE,
        model_type=nncf.ModelType.TRANSFORMER,
        subset_size=len(calibration_samples),
        fast_bias_correction=True,
    )
    quantized_model.set_rt_info(MIVOLO_INT8_REVISION, MIVOLO_INT8_REVISION_KEY)
    ov.save_model(quantized_model, MIVOLO_INT8_PATH)
    print(f"INT8 model saved to {MIVOLO_INT8_PATH}")
else:
    print(f"INT8 model is up to date: {MIVOLO_INT8_PATH}")

## 3. Validate conversion against original PyTorch models

Check MiVOLO on three held-out samples against the unpatched model. Check YOLO raw output on a deterministic synthetic input. These are numerical conversion checks, not a labeled detector accuracy evaluation.

In [ ]:
torch.manual_seed(0)
source = MiVOLOHeadExportWrapper(load_mivolo_for_export()).eval()
reference_ir = compile_for_device(core, MIVOLO_FP16_PATH, DEFAULT_DEVICE)
max_error = 0.0
with torch.inference_mode():
    for face, body, _, _ in load_fairface_samples("validation", 0, 3):
        tensor = np.concatenate([preprocess_crop(face), preprocess_crop(body)], axis=1)
        expected = source(torch.from_numpy(tensor)).cpu().numpy()
        actual = reference_ir(tensor)[reference_ir.output("head_outputs")]
        np.testing.assert_allclose(actual, expected, atol=2e-2, rtol=2e-2)
        max_error = max(max_error, float(np.max(np.abs(actual - expected))))
print(f"Original PyTorch vs FP16 IR: maximum raw-head error={max_error:.6g}")
del source, reference_ir

# Detector conversion check only: this is not a labeled detection-accuracy evaluation.
pt_path = hf_hub_download(repo_id=YOLO_REPO_ID, filename="yolov8x_person_face.pt", revision=YOLO_REVISION)
original_load = torch.load
try:
    torch.load = lambda *args, **kwargs: original_load(*args, **{**kwargs, "weights_only": False})
    detector = YOLO(pt_path).model.eval()
finally:
    torch.load = original_load
detector_input = torch.rand(1, 3, 640, 640)
with torch.inference_mode():
    detector_reference = detector(detector_input)[0].cpu().numpy()
detector_ir = compile_for_device(core, YOLO_FP16_PATH, DEFAULT_DEVICE)
detector_actual = detector_ir(detector_input.numpy())[0]
np.testing.assert_allclose(detector_actual, detector_reference, atol=0.5, rtol=0.03)
print(f"YOLO FP16 smoke check: shape={detector_actual.shape}, max error={np.max(np.abs(detector_actual - detector_reference)):.6g}")
del detector, detector_ir

## 4. Validate INT8 on held-out FairFace data

Evaluate 250 validation samples never used for calibration or accuracy control. Fail if gender accuracy drops by more than 2 percentage points or age MAE increases by more than 0.75 years. FairFace provides age buckets: midpoint MAE is a proxy, not exact-age MAE. Fixed synthetic face/body crops do not measure full-pipeline accuracy.

In [ ]:
compiled_fp16 = compile_for_device(core, MIVOLO_FP16_PATH, DEFAULT_DEVICE)
compiled_int8 = compile_for_device(core, MIVOLO_INT8_PATH, DEFAULT_DEVICE)
GENDER_DROP_LIMIT_PP = 2.0
AGE_MAE_INCREASE_LIMIT = 0.75
VALIDATION_COUNT = 250
correct_fp16 = correct_int8 = 0
errors_fp16, errors_int8 = [], []
for face, body, gender, age in load_fairface_samples("validation", 0, VALIDATION_COUNT):
    tensor = np.concatenate([preprocess_crop(face), preprocess_crop(body)], axis=1)
    fp16 = compiled_fp16(tensor)[compiled_fp16.output("head_outputs")]
    int8 = compiled_int8(tensor)[compiled_int8.output("head_outputs")]
    if not np.isfinite(fp16).all() or not np.isfinite(int8).all():
        raise ValueError("Non-finite model output during validation")
    correct_fp16 += int(np.argmax(fp16[0, :MIVOLO_NUM_GENDER_CLASSES]) == gender)
    correct_int8 += int(np.argmax(int8[0, :MIVOLO_NUM_GENDER_CLASSES]) == gender)
    errors_fp16.append(abs(float(head_outputs_to_age(fp16)[0]) - age))
    errors_int8.append(abs(float(head_outputs_to_age(int8)[0]) - age))

accuracy_fp16 = 100.0 * correct_fp16 / VALIDATION_COUNT
accuracy_int8 = 100.0 * correct_int8 / VALIDATION_COUNT
mae_fp16, mae_int8 = float(np.mean(errors_fp16)), float(np.mean(errors_int8))
print(f"Gender accuracy: FP16={accuracy_fp16:.2f}%, INT8={accuracy_int8:.2f}%")
print(f"Age bucket-midpoint MAE: FP16={mae_fp16:.3f}, INT8={mae_int8:.3f} years")
print(f"Gender drop={accuracy_fp16 - accuracy_int8:.2f} percentage points")
print(f"Age MAE increase={mae_int8 - mae_fp16:.3f} years")
if accuracy_fp16 - accuracy_int8 > GENDER_DROP_LIMIT_PP or mae_int8 - mae_fp16 > AGE_MAE_INCREASE_LIMIT:
    raise AssertionError("INT8 validation failed the configured accuracy budgets")

## 5. Interactive age and gender visualization

Select an available execution device and MiVOLO precision, then run the following
cells again after changing the selection. The pipeline letterboxes the image,
detects people/faces, applies class-wise NMS, and associates each face with a body.
If no body contains a face center, the face crop is also used as the body input.
Color conversion, normalization and output decoding run on the host; the raw IRs
are unchanged. Unsupported device/operator combinations raise compilation errors.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

device_widget = widgets.Dropdown(
    options=list(dict.fromkeys(core.available_devices + ["AUTO"])),
    value=DEFAULT_DEVICE,
    description="Device:",
)
precision_widget = widgets.Dropdown(options=["FP16", "INT8"], value="INT8", description="Precision:")
display(device_widget, precision_widget)

In [ ]:
def letterbox(image, size=(640, 640)):
    height, width = image.shape[:2]
    ratio = min(size[0] / height, size[1] / width)
    resized_size = (round(width * ratio), round(height * ratio))
    dw, dh = (size[1] - resized_size[0]) / 2, (size[0] - resized_size[1]) / 2
    resized = cv2.resize(image, resized_size, interpolation=cv2.INTER_LINEAR)
    left, top = round(dw - 0.1), round(dh - 0.1)
    padded = cv2.copyMakeBorder(
        resized, top, round(dh + 0.1), left, round(dw + 0.1),
        cv2.BORDER_CONSTANT, value=(114, 114, 114),
    )
    return padded, ratio, (left, top)


class MiVOLOPipeline:
    def __init__(self, yolo_path, mivolo_path, device):
        self.detector = compile_for_device(core, yolo_path, device)
        self.estimator = compile_for_device(core, mivolo_path, device)

    def detect(self, image, confidence=0.4, iou=0.45):
        height, width = image.shape[:2]
        padded, ratio, (left, top) = letterbox(image)
        rgb = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB)
        tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
        output = self.detector(tensor)[0][0]
        if output.ndim != 2 or output.shape[0] != 6 or not np.isfinite(output).all():
            raise ValueError(f"Expected finite person/face output [6, N], got {output.shape}")
        boxes, scores = [[], []], [[], []]
        for candidate in output.T:
            cx, cy, box_width, box_height = candidate[:4]
            x1 = max(0, min(width - 1, int((cx - box_width / 2 - left) / ratio)))
            y1 = max(0, min(height - 1, int((cy - box_height / 2 - top) / ratio)))
            x2 = max(0, min(width, int((cx + box_width / 2 - left) / ratio)))
            y2 = max(0, min(height, int((cy + box_height / 2 - top) / ratio)))
            if x2 <= x1 or y2 <= y1:
                continue
            for class_id in range(2):
                score = float(candidate[4 + class_id])
                if score >= confidence:
                    boxes[class_id].append([x1, y1, x2 - x1, y2 - y1])
                    scores[class_id].append(score)
        detections = []
        for class_boxes, class_scores in zip(boxes, scores):
            selected = []
            if class_boxes:
                indices = cv2.dnn.NMSBoxes(class_boxes, class_scores, confidence, iou)
                for index in np.asarray(indices).reshape(-1):
                    x, y, w, h = class_boxes[int(index)]
                    selected.append([x, y, x + w, y + h])
            detections.append(selected)
        return detections

    def process(self, image):
        if image is None or image.dtype != np.uint8 or image.ndim != 3 or image.shape[2] != 3:
            raise ValueError("Expected a uint8 BGR image with three channels")
        if not image.size:
            raise ValueError("Input image is empty")
        persons, faces = self.detect(image)
        predictions = []
        for face in faces:
            x1, y1, x2, y2 = face
            center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
            body = face
            for person in persons:
                if person[0] <= center_x <= person[2] and person[1] <= center_y <= person[3]:
                    body = person
                    break
            face_crop = image[y1:y2, x1:x2]
            body_crop = image[body[1]:body[3], body[0]:body[2]]
            tensor = np.concatenate([preprocess_crop(face_crop), preprocess_crop(body_crop)], axis=1)
            head = self.estimator(tensor)[self.estimator.output("head_outputs")]
            if not np.isfinite(head).all():
                raise ValueError("Non-finite MiVOLO output")
            logits = head[0, :MIVOLO_NUM_GENDER_CLASSES]
            probabilities = np.exp(logits - np.max(logits))
            probabilities /= probabilities.sum()
            gender = int(np.argmax(probabilities))
            age = float(head_outputs_to_age(head)[0])
            name = "Male" if gender == 0 else "Female"
            predictions.append({
                "face_box": face, "body_box": body,
                "label": f"Gender: {name} ({probabilities[gender] * 100:.1f}%) | Age: {age:.1f} years",
            })
        return predictions

In [ ]:
import urllib.request

image_path = BASE_DIR / "data" / "sample_portrait.jpg"
image_path.parent.mkdir(parents=True, exist_ok=True)
if not image_path.exists():
    request = urllib.request.Request(
        "https://upload.wikimedia.org/wikipedia/commons/8/8d/President_Barack_Obama.jpg",
        headers={"User-Agent": "MiVOLO-OpenVINO-Notebook/1.0"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        image_bytes = response.read()
    downloaded = cv2.imdecode(np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if downloaded is None:
        raise ValueError("Downloaded sample is not a readable image")
    image_path.write_bytes(image_bytes)

input_image = cv2.imread(str(image_path))
if input_image is None:
    raise ValueError(f"Cannot read sample image: {image_path}")
mivolo_path = MIVOLO_FP16_PATH if precision_widget.value == "FP16" else MIVOLO_INT8_PATH
pipeline = MiVOLOPipeline(YOLO_FP16_PATH, mivolo_path, device_widget.value)
predictions = pipeline.process(input_image)
annotated = input_image.copy()
for prediction in predictions:
    fx1, fy1, fx2, fy2 = prediction["face_box"]
    bx1, by1, bx2, by2 = prediction["body_box"]
    if prediction["body_box"] != prediction["face_box"]:
        cv2.rectangle(annotated, (bx1, by1), (bx2, by2), (255, 120, 0), 2)
    cv2.rectangle(annotated, (fx1, fy1), (fx2, fy2), (0, 220, 0), 2)
    print(prediction["label"])
if not predictions:
    print("No faces detected at the configured confidence threshold.")
figure, axes = plt.subplots(figsize=(12, 10))
axes.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
for prediction in predictions:
    x1, y1, x2, y2 = prediction["face_box"]
    axes.annotate(
        prediction["label"],
        xy=((x1 + x2) / 2, y1),
        xycoords="data",
        xytext=(0, -8),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=11,
        color="white",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "black", "alpha": 0.8},
    )
axes.set_title(f"OpenVINO {precision_widget.value} pipeline on {device_widget.value}")
axes.axis("off")
figure.tight_layout()
figure.savefig("screenshot.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. FP16 versus INT8 latency and throughput

Run OpenVINO's standard `benchmark_app` for both raw MiVOLO models on the selected
device, with LATENCY and THROUGHPUT benchmark hints. These hints apply only to the
measurement process; they do not rewrite the models or change the demo's compile
configuration. AUTO is passed through unchanged.

Inputs are generated by the benchmark utility. Results measure model inference,
not image preprocessing, YOLO detection, NMS, visualization, or end-to-end pipeline
latency. Both precisions use the same static input shape and duration. Actual
execution precision and device selection are reported by the utility.

In [ ]:
import subprocess
import sys
import shutil

benchmark_app = shutil.which("benchmark_app", path=str(Path(sys.executable).parent))
if benchmark_app is None:
    raise FileNotFoundError(
        "benchmark_app was not found in the notebook kernel's environment. "
        "Install OpenVINO in that environment and restart the kernel."
    )
target_device = device_widget.value
for precision, path in (("FP16", MIVOLO_FP16_PATH), ("INT8", MIVOLO_INT8_PATH)):
    for hint in ("latency", "throughput"):
        print(f"\n--- MiVOLO {precision}: {target_device}, {hint}, 10 seconds ---", flush=True)
        subprocess.run(
            [
                benchmark_app,
                "-m", str(path), "-d", target_device,
                "-api", "async", "-hint", hint, "-t", "10",
            ],
            check=True,
        )